In [22]:
import importlib
import pandas as pd
import numpy as np
import json
import folium

#Mandatory
importlib.reload(importlib)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
main_tracks_df = pd.read_csv(f"sumo_data/main_tracks.csv", sep=";")
# main_tracks_df = pd.read_csv(f"cleaned_data/main_tracks.csv", sep=";")

In [24]:
main_tracks_df.info()
main_tracks_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26713 entries, 0 to 26712
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                26713 non-null  int64  
 1   Departure_switch  26713 non-null  int64  
 2   Arrival_switch    26713 non-null  int64  
 3   Path              26713 non-null  object 
 4   Length_m          26713 non-null  float64
dtypes: float64(1), int64(3), object(1)
memory usage: 1.0+ MB


,ID,Departure_switch,Arrival_switch,Path,Length_m
0,0,0,1,"[[50.816795, 4.395716, 0.0], [50.816754, 4.395...",685.820687
1,1,0,2,"[[50.816795, 4.395716, 0.0], [50.816429, 4.396...",644.003047
2,2,3,4,"[[50.810215, 4.398703, 88.85], [50.810358, 4.3...",62.034102
3,3,4,1,"[[50.810734, 4.399025, 89.02], [50.81096, 4.39...",57.621587
4,4,4,5,"[[50.810734, 4.399025, 89.02], [50.810778, 4.3...",267.665967


In [25]:
def parse_coordinates(coord_str):
	try:
		coords = json.loads(coord_str)
		if isinstance(coords, list) and all(isinstance(c, list) for c in coords):
			return [[float(x) for x in c] for c in coords]
	except Exception:
		print(f"Invalid coordinate format: {coord_str}")
	return np.nan

# main_tracks_df["Coordinates"] = main_tracks_df["Coordinates"].apply(parse_coordinates)
main_tracks_df["Path"] = main_tracks_df["Path"].apply(parse_coordinates)

In [26]:
graph = {}
for index, row in main_tracks_df.iterrows():
	# coordinates = row["Coordinates"]
	coordinates = row["Path"]
	start = tuple([coordinates[0][0], coordinates[0][1]])
	end = tuple([coordinates[-1][0], coordinates[-1][1]])
	if start not in graph :
		graph[start] = set()
	graph[start].add(end)
	if end not in graph :
		graph[end] = set()
	graph[end].add(start)
	# for i in range(len(coordinates) - 1) :
	# 	start = tuple(coordinates[i])
	# 	end = tuple(coordinates[i + 1])
	# 	if start not in graph :
	# 		graph[start] = set()
	# 	graph[start].add(end)
	# 	if end not in graph :
	# 		graph[end] = set()
	# 	graph[end].add(start)

# for node in graph:
# 	if len(graph[node]) > 4:
# 		print(f"{node} -> {graph[node]}")

In [27]:
for node in graph:
	print(f"{node} -> {graph[node]}")

(50.816795, 4.395716) -> {(50.811235, 4.399232), (50.811562, 4.399146), (50.817152, 4.395299)}
(50.811235, 4.399232) -> {(50.816795, 4.395716), (50.810734, 4.399025)}
(50.811562, 4.399146) -> {(50.816795, 4.395716), (50.811094, 4.399312)}
(50.810215, 4.398703) -> {(50.817486, 4.394845), (50.810734, 4.399025), (50.807441, 4.396978)}
(50.810734, 4.399025) -> {(50.810246, 4.398781), (50.811235, 4.399232), (50.812973, 4.400416), (50.810215, 4.398703)}
(50.812973, 4.400416) -> {(50.813224, 4.400571), (50.810734, 4.399025)}
(51.284903, 4.222322) -> {(51.284933, 4.215361), (51.284965, 4.216122), (51.286603, 4.226316)}
(51.286603, 4.226316) -> {(51.284903, 4.222322), (51.287435, 4.227325), (51.286384, 4.226082)}
(51.14025, 3.79489) -> {(51.137715, 3.792106), (51.140595, 3.795356), (51.143568, 3.792761), (51.13873, 3.786968)}
(51.13873, 3.786968) -> {(51.14025, 3.79489)}
(51.181114, 3.574866) -> {(51.178, 3.578964)}
(51.178, 3.578964) -> {(51.17551, 3.582715), (51.181114, 3.574866), (51.178602,

In [28]:
zero_link_switches = sum(1 for node in graph if len(graph[node]) == 0)
one_link_switches = sum(1 for node in graph if len(graph[node]) == 1)
two_link_switches = sum(1 for node in graph if len(graph[node]) == 2)
three_link_switches = sum(1 for node in graph if len(graph[node]) == 3)
four_link_switches = sum(1 for node in graph if len(graph[node]) == 4)
n_link_switches = sum(1 for node in graph if len(graph[node]) > 4)
print(f"total switches : {len(graph)}")
print(f"switches with zero link : {zero_link_switches} ({zero_link_switches / len(graph) * 100:.2f}%)")
print(f"switches with one link : {one_link_switches} ({one_link_switches / len(graph) * 100:.2f}%)")
print(f"switches with two links : {two_link_switches} ({two_link_switches / len(graph) * 100:.2f}%)")
print(f"switches with three links : {three_link_switches} ({three_link_switches / len(graph) * 100:.2f}%)")
print(f"switches with four links : {four_link_switches} ({four_link_switches / len(graph) * 100:.2f}%)")
print(f"switches with more than four links : {n_link_switches} ({n_link_switches / len(graph) * 100:.2f}%)")

total switches : 22061
switches with zero link : 0 (0.00%)
switches with one link : 4539 (20.57%)
switches with two links : 6082 (27.57%)
switches with three links : 10126 (45.90%)
switches with four links : 1308 (5.93%)
switches with more than four links : 6 (0.03%)


In [29]:
nodes_id = {i : node for i, node in enumerate(graph.keys())}
for i in range(5):
	print(f"Node ID {i}: {nodes_id[i]})")

Node ID 0: (50.816795, 4.395716))
Node ID 1: (50.811235, 4.399232))
Node ID 2: (50.811562, 4.399146))
Node ID 3: (50.810215, 4.398703))
Node ID 4: (50.810734, 4.399025))


In [30]:
links_per_node = {node_id : len(graph[nodes_id[node_id]]) for node_id in nodes_id.keys()}
for i in range(5):
	print(f"Number of links for node {i}: {links_per_node[i]})")

Number of links for node 0: 3)
Number of links for node 1: 2)
Number of links for node 2: 2)
Number of links for node 3: 3)
Number of links for node 4: 4)


In [31]:
switches = [node_id for node_id in links_per_node.keys() if links_per_node[node_id] >= 3]
for sid in switches[:5]:
	print(f"Switch Node ID: {sid}, Coordinates: {nodes_id[sid]}, Number of links: {links_per_node[sid]}")

Switch Node ID: 0, Coordinates: (50.816795, 4.395716), Number of links: 3
Switch Node ID: 3, Coordinates: (50.810215, 4.398703), Number of links: 3
Switch Node ID: 4, Coordinates: (50.810734, 4.399025), Number of links: 4
Switch Node ID: 6, Coordinates: (51.284903, 4.222322), Number of links: 3
Switch Node ID: 7, Coordinates: (51.286603, 4.226316), Number of links: 3


In [32]:
for sid in switches :
	node = nodes_id[sid]
	links = graph[node]
	path = [node]

In [33]:
m = folium.Map(location=[50.8477, 4.3572], zoom_start=12)
for index, row in main_tracks_df.iterrows():
	# coordinates = row["Coordinates"]
	coordinates = row["Path"]
	folium.PolyLine(locations=[(coord[0], coord[1]) for coord in coordinates], color='black', weight=2).add_to(m)
for node in graph:
	if len(graph[node]) == 1:
		folium.CircleMarker(location=[node[0], node[1]], radius=5, color='black', fill=True).add_to(m)
	if len(graph[node]) == 2:
		folium.CircleMarker(location=[node[0], node[1]], radius=5, color='blue', fill=True).add_to(m)
	if len(graph[node]) == 3:
		folium.CircleMarker(location=[node[0], node[1]], radius=5, color='green', fill=True).add_to(m)
	if len(graph[node]) == 4:
		folium.CircleMarker(location=[node[0], node[1]], radius=5, color='red', fill=True).add_to(m)
	if len(graph[node]) > 4:
		folium.CircleMarker(location=[node[0], node[1]], radius=5, color='pink', fill=True).add_to(m)
# m.save("switches_map.html")
m.save("simplified_switches_map.html")

In [34]:
import pandas as pd
import folium
import json
from branca.element import Element

def plot_tracks_interactive(df, output_file="railway_map.html",
                            default_color="black", selected_color="red",
                            line_weight=4, zoom_start=12):
    """
    Affiche les segments du dataframe sur une carte Folium.
    
    Paramètres
    ----------
    df : pandas.DataFrame
        Doit contenir les colonnes:
        - 'ID'
        - 'Coordinates' : liste de points [[lat, lon, z], [lat, lon, z], ...]
          ou chaîne JSON équivalente.
    output_file : str
        Nom du fichier HTML généré.
    default_color : str
        Couleur par défaut des lignes.
    selected_color : str
        Couleur après clic.
    line_weight : int
        Épaisseur des lignes.
    zoom_start : int
        Niveau de zoom initial.
    """

    def parse_coordinates(coords):
        # Si la colonne contient une chaîne, on tente de la parser
        if isinstance(coords, str):
            coords = json.loads(coords)

        # On garde uniquement [lat, lon]
        parsed = []
        for p in coords:
            if len(p) >= 2:
                lat, lon = p[0], p[1]
                parsed.append([lat, lon])
        return parsed

    # Préparation des segments
    features = []
    all_points = []

    for _, row in df.iterrows():
        # coords = parse_coordinates(row["Coordinates"])
        coords = parse_coordinates(row["Path"])
        if len(coords) < 2:
            continue

        start_point = coords[0]
        end_point = coords[-1]

        all_points.extend(coords)

        feature = {
            "type": "Feature",
            "properties": {
                "id": row["ID"],
                "start_lat": start_point[0],
                "start_lon": start_point[1],
                "end_lat": end_point[0],
                "end_lon": end_point[1],
            },
            "geometry": {
                "type": "LineString",
                # GeoJSON attend [lon, lat]
                "coordinates": [[pt[1], pt[0]] for pt in coords]
            }
        }
        features.append(feature)

    if not all_points:
        raise ValueError("Aucune géométrie valide trouvée dans le dataframe.")

    # Centre de la carte
    center_lat = sum(p[0] for p in all_points) / len(all_points)
    center_lon = sum(p[1] for p in all_points) / len(all_points)

    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start)

    geojson_data = {
        "type": "FeatureCollection",
        "features": features
    }

    # Couche GeoJSON
    g = folium.GeoJson(
        geojson_data,
        name="tracks",
        style_function=lambda feature: {
            "color": default_color,
            "weight": line_weight,
            "opacity": 0.8
        },
        popup=folium.GeoJsonPopup(
            fields=["id", "start_lat", "start_lon", "end_lat", "end_lon"],
            aliases=["ID", "Start lat", "Start lon", "End lat", "End lon"],
            localize=True,
            labels=True
        )
    )
    g.add_to(m)

    # JS personnalisé : au clic, la ligne cliquée devient rouge
    # et les autres reviennent à la couleur par défaut
    map_name = m.get_name()
    geojson_name = g.get_name()

    custom_js = f"""
    <script>
    var selectedLayer = null;

    function resetGeoJsonStyles() {{
        {geojson_name}.eachLayer(function(layer) {{
            layer.setStyle({{
                color: "{default_color}",
                weight: {line_weight},
                opacity: 0.8
            }});
        }});
    }}

    {geojson_name}.eachLayer(function(layer) {{
        layer.on('click', function(e) {{
            resetGeoJsonStyles();

            e.target.setStyle({{
                color: "{selected_color}",
                weight: {line_weight + 2},
                opacity: 1.0
            }});

            selectedLayer = e.target;
        }});
    }});
    </script>
    """
    m.get_root().html.add_child(Element(custom_js))

    folium.LayerControl().add_to(m)
    m.save(output_file)

    return m

In [35]:
# Exemple :
# df = ton dataframe pandas

# railway_m = plot_tracks_interactive(main_tracks_df, output_file="railway_map.html")
railway_m = plot_tracks_interactive(main_tracks_df, output_file="simplified_railway_map.html")